# TUM Student Performance — Model Training
Trains a Random Forest classifier to predict final exam outcome
(Pass / At Risk / Fail) using ONLY pre-exam features:
- Attendance %
- CAT1 score (out of 30)
- CAT2 score (out of 30)
- CAT average
- Assignment score %
- Assignments submitted %

Final exam score is NOT a feature — this enables early intervention.


In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/student_performance.csv')
print('Dataset shape:', df.shape)
print('Class distribution:')
print(df['Performance'].value_counts())

In [ ]:
# Features: NO Final_Exam — enables prediction before end of semester
FEATURES = [
    'Attendance',
    'CAT1_Score',
    'CAT2_Score',
    'CAT_Average',
    'Assignment_Score',
    'Assignments_Submitted'
]

X = df[FEATURES]
y = df['Performance']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight='balanced'  # handles imbalanced At Risk class
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print('Classification Report:')
print(classification_report(y_test, y_pred))

In [ ]:
cv_scores = cross_val_score(model, X, y, cv=5)
print(f'Cross-validation accuracy: {cv_scores.mean():.2f} (+/- {cv_scores.std():.2f})')

print('\nFeature importances:')
for feat, imp in sorted(zip(FEATURES, model.feature_importances_), key=lambda x: -x[1]):
    print(f'  {feat}: {imp:.3f}')

In [ ]:
joblib.dump(model, '../models/student_performance_model.pkl')
print('Model saved to models/student_performance_model.pkl')
print('Features:', FEATURES)
print('Classes:', list(model.classes_))